# 03. Full-topology reference systems at reduced tensor scale

This notebook uses **canonical reference variants** so architectural fidelity can be checked mechanically.

Only tensor scale is reduced: hidden width, head width, batch size, token count, image resolution, vocabulary size, and training budget.

The following topology is kept unchanged:

- GPT-2 small: 12 decoder blocks, 12 query heads, learned absolute positions, pre-LN residual blocks, tied LM head.
- ViT-B/16: 12 encoder blocks, 12 heads, 16×16 patch embedding, CLS token, learned position embedding, MLP ratio 4.
- DiT-B/2: 12 DiT blocks, 12 heads, patch size 2, adaLN-Zero on attention/MLP, class+timestep conditioning, modulated final layer.
- π0: 27-layer SigLIP-style image tower and an 18-layer Gemma/action-expert joint stack, GQA, RoPE, block attention mask, flow matching.

No layer count or branch is removed.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device("cpu")
print("device:", device)

## 1. Shared readable Transformer primitives

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        assert hidden_dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.output = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, causal=True):
        batch_size, sequence_length, hidden_dim = x.shape

        qkv = self.qkv(x)
        qkv = qkv.view(
            batch_size,
            sequence_length,
            3,
            self.num_heads,
            self.head_dim,
        )
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)

        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=causal,
        )
        attended = attended.transpose(1, 2).contiguous()
        attended = attended.view(batch_size, sequence_length, hidden_dim)

        return self.output(attended)


class GPT2Block(nn.Module):
    def __init__(self, hidden_dim=48, num_heads=12):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attention = CausalSelfAttention(hidden_dim, num_heads)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x), causal=True)
        x = x + self.mlp(self.norm2(x))
        return x

## 2. GPT-2 small topology: 12 decoder blocks

In [ ]:
class SmallWidthGPT2(nn.Module):
    def __init__(
        self,
        vocab_size=128,
        max_length=32,
        hidden_dim=48,
        num_heads=12,
        depth=12,
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.position_embedding = nn.Embedding(max_length, hidden_dim)

        self.blocks = nn.ModuleList(
            [
                GPT2Block(hidden_dim, num_heads)
                for _ in range(depth)
            ]
        )
        self.final_norm = nn.LayerNorm(hidden_dim)

    def forward(self, token_ids):
        sequence_length = token_ids.size(1)
        position_ids = torch.arange(
            sequence_length,
            device=token_ids.device,
        )

        hidden = (
            self.token_embedding(token_ids)
            + self.position_embedding(position_ids)[None]
        )

        for block in self.blocks:
            hidden = block(hidden)

        hidden = self.final_norm(hidden)
        logits = F.linear(hidden, self.token_embedding.weight)
        return logits


gpt = SmallWidthGPT2().to(device)

assert len(gpt.blocks) == 12
assert gpt.blocks[0].attention.num_heads == 12

token_ids = torch.randint(0, 128, (1, 8), device=device)
gpt_logits = gpt(token_ids[:, :-1])
gpt_loss = F.cross_entropy(
    gpt_logits.reshape(-1, 128),
    token_ids[:, 1:].reshape(-1),
)
gpt_loss.backward()

print("GPT-2 blocks:", len(gpt.blocks))
print("GPT-2 heads:", gpt.blocks[0].attention.num_heads)
print("GPT-2 loss:", gpt_loss.item())

## 3. ViT-B/16 topology: 12 encoder blocks

In [ ]:
class ViTBlock(nn.Module):
    def __init__(self, hidden_dim=48, num_heads=12):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attention = nn.MultiheadAttention(
            hidden_dim,
            num_heads,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

    def forward(self, x):
        normalized = self.norm1(x)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            need_weights=False,
        )
        x = x + attended
        x = x + self.mlp(self.norm2(x))
        return x


class SmallWidthViTB16(nn.Module):
    def __init__(
        self,
        image_size=32,
        patch_size=16,
        hidden_dim=48,
        num_heads=12,
        depth=12,
        classes=10,
    ):
        super().__init__()

        assert image_size % patch_size == 0

        self.patch_size = patch_size
        self.patch_projection = nn.Conv2d(
            3,
            hidden_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

        patch_count = (image_size // patch_size) ** 2

        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, hidden_dim)
        )
        self.position_embedding = nn.Parameter(
            torch.randn(1, patch_count + 1, hidden_dim) * 0.02
        )

        self.blocks = nn.ModuleList(
            [
                ViTBlock(hidden_dim, num_heads)
                for _ in range(depth)
            ]
        )
        self.final_norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, classes)

    def forward(self, image):
        patches = self.patch_projection(image)
        patches = patches.flatten(2).transpose(1, 2)

        cls = self.cls_token.expand(image.size(0), -1, -1)
        hidden = torch.cat([cls, patches], dim=1)
        hidden = hidden + self.position_embedding[:, : hidden.size(1)]

        for block in self.blocks:
            hidden = block(hidden)

        cls_hidden = self.final_norm(hidden[:, 0])
        return self.classifier(cls_hidden)


vit = SmallWidthViTB16().to(device)

assert len(vit.blocks) == 12
assert vit.patch_size == 16
assert vit.blocks[0].attention.num_heads == 12

image = torch.randn(1, 3, 32, 32, device=device)
label = torch.tensor([3], device=device)

vit_loss = F.cross_entropy(vit(image), label)
vit_loss.backward()

print("ViT blocks:", len(vit.blocks))
print("ViT patch size:", vit.patch_size)
print("ViT loss:", vit_loss.item())

## 4. DiT-B/2 topology: 12 blocks + adaLN-Zero + modulated final layer

In [ ]:
def sinusoidal_timestep_embedding(timestep, dim):
    half = dim // 2

    frequencies = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=timestep.device)
        / max(half - 1, 1)
    )
    angles = timestep[:, None] * frequencies[None]

    return torch.cat(
        [angles.cos(), angles.sin()],
        dim=-1,
    )


def modulate(x, shift, scale):
    return x * (1 + scale[:, None, :]) + shift[:, None, :]


class DiTBlock(nn.Module):
    def __init__(self, hidden_dim=48, num_heads=12):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.attention = nn.MultiheadAttention(
            hidden_dim,
            num_heads,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

        self.adaLN = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 6 * hidden_dim),
        )
        nn.init.zeros_(self.adaLN[-1].weight)
        nn.init.zeros_(self.adaLN[-1].bias)

    def forward(self, hidden, condition):
        (
            shift_attn,
            scale_attn,
            gate_attn,
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.adaLN(condition).chunk(6, dim=-1)

        attention_input = modulate(
            self.norm1(hidden),
            shift_attn,
            scale_attn,
        )
        attention_output, _ = self.attention(
            attention_input,
            attention_input,
            attention_input,
            need_weights=False,
        )
        hidden = (
            hidden
            + gate_attn[:, None, :] * attention_output
        )

        mlp_input = modulate(
            self.norm2(hidden),
            shift_mlp,
            scale_mlp,
        )
        hidden = (
            hidden
            + gate_mlp[:, None, :] * self.mlp(mlp_input)
        )

        return hidden


class DiTFinalLayer(nn.Module):
    def __init__(self, hidden_dim, patch_size, channels):
        super().__init__()

        self.norm = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.adaLN = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 2 * hidden_dim),
        )
        self.linear = nn.Linear(
            hidden_dim,
            patch_size * patch_size * channels,
        )

        nn.init.zeros_(self.adaLN[-1].weight)
        nn.init.zeros_(self.adaLN[-1].bias)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, hidden, condition):
        shift, scale = self.adaLN(condition).chunk(2, dim=-1)
        hidden = modulate(self.norm(hidden), shift, scale)
        return self.linear(hidden)


class SmallWidthDiTB2(nn.Module):
    def __init__(
        self,
        image_size=8,
        patch_size=2,
        channels=1,
        hidden_dim=48,
        num_heads=12,
        depth=12,
        num_classes=10,
    ):
        super().__init__()

        self.image_size = image_size
        self.patch_size = patch_size
        self.channels = channels

        self.patch_projection = nn.Conv2d(
            channels,
            hidden_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

        token_count = (image_size // patch_size) ** 2
        self.position_embedding = nn.Parameter(
            torch.randn(1, token_count, hidden_dim) * 0.02
        )

        self.time_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.SiLU(),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )
        self.class_embedding = nn.Embedding(
            num_classes + 1,
            hidden_dim,
        )

        self.blocks = nn.ModuleList(
            [
                DiTBlock(hidden_dim, num_heads)
                for _ in range(depth)
            ]
        )
        self.final_layer = DiTFinalLayer(
            hidden_dim,
            patch_size,
            channels,
        )

    def forward(self, image, timestep, class_id):
        hidden = self.patch_projection(image)
        hidden = hidden.flatten(2).transpose(1, 2)
        hidden = hidden + self.position_embedding

        time_embedding = sinusoidal_timestep_embedding(
            timestep,
            hidden.size(-1),
        )
        condition = (
            self.time_mlp(time_embedding)
            + self.class_embedding(class_id)
        )

        for block in self.blocks:
            hidden = block(hidden, condition)

        patches = self.final_layer(hidden, condition)
        patches = patches.transpose(1, 2)

        return F.fold(
            patches,
            output_size=(self.image_size, self.image_size),
            kernel_size=self.patch_size,
            stride=self.patch_size,
        )


dit = SmallWidthDiTB2().to(device)

assert len(dit.blocks) == 12
assert dit.patch_size == 2
assert dit.blocks[0].attention.num_heads == 12

clean = torch.randn(1, 1, 8, 8, device=device)
noise = torch.randn_like(clean)
timestep = torch.rand(1, device=device)
class_id = torch.tensor([2], device=device)

alpha = torch.cos(0.5 * math.pi * timestep)
sigma = torch.sin(0.5 * math.pi * timestep)

noisy = (
    alpha[:, None, None, None] * clean
    + sigma[:, None, None, None] * noise
)
epsilon_prediction = dit(noisy, timestep, class_id)
dit_loss = F.mse_loss(epsilon_prediction, noise)
dit_loss.backward()

print("DiT blocks:", len(dit.blocks))
print("DiT patch size:", dit.patch_size)
print("DiT loss:", dit_loss.item())

## 5. π0 topology: 27-layer vision tower + 18-layer joint base/action expert

In [ ]:
def apply_rope(x, position_ids, base=10000.0):
    head_dim = x.size(-1)
    assert head_dim % 2 == 0

    pair_index = torch.arange(
        0,
        head_dim,
        2,
        device=x.device,
        dtype=torch.float32,
    )
    inverse_frequency = 1.0 / (
        base ** (pair_index / head_dim)
    )
    angle = (
        position_ids.float()[..., None]
        * inverse_frequency
    )

    if x.ndim == 4:
        angle = angle[:, None, :, :]
    elif x.ndim != 3:
        raise ValueError(
            f"RoPE expects [B,S,D] or [B,H,S,D], got {tuple(x.shape)}"
        )

    cosine = angle.cos()
    sine = angle.sin()

    even = x[..., 0::2]
    odd = x[..., 1::2]

    rotated_even = even * cosine - odd * sine
    rotated_odd = even * sine + odd * cosine

    return torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    ).flatten(-2)


class VisionTransformerBlock(nn.Module):
    def __init__(self, hidden_dim=64, num_heads=16):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attention = nn.MultiheadAttention(
            hidden_dim,
            num_heads,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

    def forward(self, x):
        normalized = self.norm1(x)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            need_weights=False,
        )
        x = x + attended
        x = x + self.mlp(self.norm2(x))
        return x


class SmallWidthSigLIPSo400mTower(nn.Module):
    def __init__(
        self,
        image_size=28,
        patch_size=14,
        hidden_dim=64,
        depth=27,
        num_heads=16,
        output_dim=32,
    ):
        super().__init__()

        self.patch_size = patch_size
        self.patch = nn.Conv2d(
            3,
            hidden_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

        patch_count = (image_size // patch_size) ** 2
        self.position = nn.Parameter(
            torch.randn(1, patch_count, hidden_dim) * 0.02
        )

        self.blocks = nn.ModuleList(
            [
                VisionTransformerBlock(hidden_dim, num_heads)
                for _ in range(depth)
            ]
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.projection = nn.Linear(hidden_dim, output_dim)

    def forward(self, image):
        hidden = self.patch(image).flatten(2).transpose(1, 2)
        hidden = hidden + self.position

        for block in self.blocks:
            hidden = block(hidden)

        return self.projection(self.norm(hidden))


def make_block_attention_mask(input_mask, block_start):
    block_start = block_start.expand_as(input_mask)
    block_id = torch.cumsum(block_start.to(torch.long), dim=1)

    can_attend = (
        block_id[:, None, :]
        <= block_id[:, :, None]
    )
    valid = input_mask[:, None, :] & input_mask[:, :, None]
    return can_attend & valid


def pi0_time_embedding(timestep, dim):
    half = dim // 2
    fraction = torch.linspace(
        0.0,
        1.0,
        half,
        device=timestep.device,
    )
    periods = 4e-3 * (4.0 / 4e-3) ** fraction
    angles = (
        timestep[:, None]
        / periods[None]
        * 2.0
        * math.pi
    )
    return torch.cat(
        [angles.sin(), angles.cos()],
        dim=-1,
    )


class GQAJointBlock(nn.Module):
    def __init__(
        self,
        hidden_dim=32,
        query_heads=8,
        kv_heads=1,
    ):
        super().__init__()

        assert hidden_dim % query_heads == 0
        assert query_heads % kv_heads == 0

        self.hidden_dim = hidden_dim
        self.query_heads = query_heads
        self.kv_heads = kv_heads
        self.head_dim = hidden_dim // query_heads

        self.base_norm1 = nn.RMSNorm(hidden_dim)
        self.base_q = nn.Linear(
            hidden_dim,
            query_heads * self.head_dim,
            bias=False,
        )
        self.base_k = nn.Linear(
            hidden_dim,
            kv_heads * self.head_dim,
            bias=False,
        )
        self.base_v = nn.Linear(
            hidden_dim,
            kv_heads * self.head_dim,
            bias=False,
        )
        self.base_out = nn.Linear(
            query_heads * self.head_dim,
            hidden_dim,
            bias=False,
        )

        self.expert_norm1 = nn.RMSNorm(hidden_dim)
        self.expert_q = nn.Linear(
            hidden_dim,
            query_heads * self.head_dim,
            bias=False,
        )
        self.expert_k = nn.Linear(
            hidden_dim,
            kv_heads * self.head_dim,
            bias=False,
        )
        self.expert_v = nn.Linear(
            hidden_dim,
            kv_heads * self.head_dim,
            bias=False,
        )
        self.expert_out = nn.Linear(
            query_heads * self.head_dim,
            hidden_dim,
            bias=False,
        )

        self.base_norm2 = nn.RMSNorm(hidden_dim)
        self.base_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

        self.expert_norm2 = nn.RMSNorm(hidden_dim)
        self.expert_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

    def _project_q(self, hidden, projection):
        batch_size, sequence_length, _ = hidden.shape
        q = projection(hidden)
        return q.view(
            batch_size,
            sequence_length,
            self.query_heads,
            self.head_dim,
        ).transpose(1, 2)

    def _project_kv(self, hidden, projection):
        batch_size, sequence_length, _ = hidden.shape
        kv = projection(hidden)
        kv = kv.view(
            batch_size,
            sequence_length,
            self.kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        repeats = self.query_heads // self.kv_heads
        return kv.repeat_interleave(repeats, dim=1)

    def forward(
        self,
        prefix,
        suffix,
        attention_mask,
        position_ids,
    ):
        prefix_norm = self.base_norm1(prefix)
        suffix_norm = self.expert_norm1(suffix)

        prefix_q = self._project_q(prefix_norm, self.base_q)
        prefix_k = self._project_kv(prefix_norm, self.base_k)
        prefix_v = self._project_kv(prefix_norm, self.base_v)

        suffix_q = self._project_q(suffix_norm, self.expert_q)
        suffix_k = self._project_kv(suffix_norm, self.expert_k)
        suffix_v = self._project_kv(suffix_norm, self.expert_v)

        q = torch.cat([prefix_q, suffix_q], dim=2)
        k = torch.cat([prefix_k, suffix_k], dim=2)
        v = torch.cat([prefix_v, suffix_v], dim=2)

        q = apply_rope(q, position_ids)
        k = apply_rope(k, position_ids)

        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask[:, None],
        )

        prefix_length = prefix.size(1)

        prefix_attended = attended[:, :, :prefix_length]
        suffix_attended = attended[:, :, prefix_length:]

        prefix_attended = prefix_attended.transpose(1, 2)
        prefix_attended = prefix_attended.contiguous().flatten(2)

        suffix_attended = suffix_attended.transpose(1, 2)
        suffix_attended = suffix_attended.contiguous().flatten(2)

        prefix = prefix + self.base_out(prefix_attended)
        suffix = suffix + self.expert_out(suffix_attended)

        prefix = prefix + self.base_mlp(self.base_norm2(prefix))
        suffix = suffix + self.expert_mlp(self.expert_norm2(suffix))

        return prefix, suffix


class SmallTensorPi0(nn.Module):
    def __init__(
        self,
        hidden_dim=32,
        action_dim=3,
        action_horizon=4,
        vocabulary_size=128,
    ):
        super().__init__()

        self.action_dim = action_dim
        self.action_horizon = action_horizon

        self.vision_tower = SmallWidthSigLIPSo400mTower(
            output_dim=hidden_dim,
        )
        self.language_embedding = nn.Embedding(
            vocabulary_size,
            hidden_dim,
        )

        self.state_projection = nn.Linear(
            6,
            hidden_dim,
        )
        self.action_projection = nn.Linear(
            action_dim,
            hidden_dim,
        )
        self.action_time_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.joint_layers = nn.ModuleList(
            [
                GQAJointBlock(
                    hidden_dim=hidden_dim,
                    query_heads=8,
                    kv_heads=1,
                )
                for _ in range(18)
            ]
        )
        self.action_output = nn.Linear(
            hidden_dim,
            action_dim,
        )

    def forward(
        self,
        image,
        language_ids,
        state,
        noisy_actions,
        timestep,
    ):
        image_tokens = self.vision_tower(image)
        language_tokens = self.language_embedding(language_ids)

        prefix = torch.cat(
            [image_tokens, language_tokens],
            dim=1,
        )

        state_token = self.state_projection(state).unsqueeze(1)

        action_tokens = self.action_projection(noisy_actions)
        time_embedding = pi0_time_embedding(
            timestep,
            action_tokens.size(-1),
        )
        time_tokens = time_embedding[:, None].expand_as(action_tokens)
        action_tokens = self.action_time_mlp(
            torch.cat(
                [action_tokens, time_tokens],
                dim=-1,
            )
        )

        suffix = torch.cat(
            [state_token, action_tokens],
            dim=1,
        )

        prefix_mask = torch.ones(
            prefix.size(0),
            prefix.size(1),
            dtype=torch.bool,
            device=prefix.device,
        )
        suffix_mask = torch.ones(
            suffix.size(0),
            suffix.size(1),
            dtype=torch.bool,
            device=prefix.device,
        )
        input_mask = torch.cat(
            [prefix_mask, suffix_mask],
            dim=1,
        )

        block_start = torch.cat(
            [
                torch.zeros(
                    prefix.size(1),
                    dtype=torch.bool,
                    device=prefix.device,
                ),
                torch.tensor(
                    [True] + [True] + [False] * (self.action_horizon - 1),
                    dtype=torch.bool,
                    device=prefix.device,
                ),
            ]
        )[None]

        attention_mask = make_block_attention_mask(
            input_mask,
            block_start,
        )
        position_ids = (
            torch.cumsum(input_mask.to(torch.long), dim=1)
            - 1
        )

        for layer in self.joint_layers:
            prefix, suffix = layer(
                prefix,
                suffix,
                attention_mask,
                position_ids,
            )

        action_hidden = suffix[:, -self.action_horizon:]
        return self.action_output(action_hidden)


pi0 = SmallTensorPi0().to(device)

assert len(pi0.vision_tower.blocks) == 27
assert len(pi0.joint_layers) == 18
assert pi0.joint_layers[0].query_heads == 8
assert pi0.joint_layers[0].kv_heads == 1

pi0_image = torch.randn(1, 3, 28, 28, device=device)
pi0_language = torch.randint(0, 128, (1, 3), device=device)
pi0_state = torch.randn(1, 6, device=device)
pi0_actions = torch.randn(1, 4, 3, device=device)
pi0_noise = torch.randn_like(pi0_actions)
pi0_t = torch.rand(1, device=device) * 0.998 + 0.001

pi0_xt = (
    pi0_t[:, None, None] * pi0_noise
    + (1 - pi0_t[:, None, None]) * pi0_actions
)
pi0_target = pi0_noise - pi0_actions

pi0_velocity = pi0(
    pi0_image,
    pi0_language,
    pi0_state,
    pi0_xt,
    pi0_t,
)
pi0_loss = F.mse_loss(pi0_velocity, pi0_target)
pi0_loss.backward()

print("pi0 vision depth:", len(pi0.vision_tower.blocks))
print("pi0 joint depth:", len(pi0.joint_layers))
print("pi0 flow loss:", pi0_loss.item())

## Architecture audit

The assertions above are intentional. If a later edit silently replaces a reference architecture with a shallower surrogate model, this notebook fails immediately.

References:

- GPT-2: 12-block small decoder topology.
- ViT: ViT-B/16 encoder topology.
- DiT: DiT-B/2 block topology and adaLN-Zero/final-layer modulation.
- Physical Intelligence openpi: π0 prefix/suffix block mask, sinusoidal time embedding, flow-matching target, SigLIP/PaliGemma + action-expert structure.